# Bootstrap para Intervalos de Confianza
### Métodos Percentil, BCa e Implementaciones

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Este notebook es la versión **ejecutable** del apunte. Cada método bootstrap (Normal, Percentil, BCa) se implementa desde cero y se ejecuta de verdad; además **convertimos las ideas en demostraciones**: simulamos el significado real de un intervalo de confianza, dibujamos la distribución bootstrap con sus límites, superponemos los tres métodos sobre los mismos datos y medimos la **cobertura empírica** de cada uno para comprobar cuál funciona mejor.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar.

**Requisitos:** sólo `numpy`, `matplotlib` y `scipy` — todo viene preinstalado en Google Colab, así que corre sin instalar nada.

## Configuración del entorno

Importamos librerías y fijamos un estilo de gráficos. **Ejecuta esta celda primero.** Las funciones que definamos más abajo quedan disponibles para las celdas siguientes, así que **ejecuta el notebook en orden** (o usa *Run All*).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import bootstrap

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titleweight"] = "bold"

import scipy
print("Entorno listo.  numpy", np.__version__, "| scipy", scipy.__version__)

# 1. Introducción: Del Remuestreo a los Intervalos

En el apunte anterior aprendimos el concepto básico de remuestreo. Ahora abordamos la aplicación más importante del bootstrap: **construir intervalos de confianza**.

Tienes datos $\mathbf{x} = (x_1,\ldots,x_n)$ y calculas un estadístico $\hat{\theta} = T(\mathbf{x})$ (media, mediana, correlación, etc.). La pregunta central es:

> *¿Qué tan precisa es mi estimación $\hat{\theta}$? ¿En qué rango es probable que esté el parámetro verdadero $\theta$?*

El **enfoque paramétrico** requiere conocer la distribución de $\hat{\theta}$: para la media normal aplica $\bar{x} \pm t_{\alpha/2}\cdot s/\sqrt{n}$, pero no funciona para estadísticos complejos como la mediana, el percentil 90 o la correlación de Spearman. El **enfoque bootstrap** no requiere supuestos distribucionales: usa remuestreo para estimar la distribución empírica de $\hat{\theta}$ y construye el IC directamente a partir de sus percentiles.

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 3))
ax.set_xlim(0, 13); ax.set_ylim(0, 3); ax.axis("off")

etapas = [
    (1.4,  "Datos",                  r"$\mathbf{x}$, $n$ obs.",                       "#cdd6e0"),
    (4.7,  "Bootstrap",              r"$B$ remuestras" "\n" r"$\hat{\theta}^*_1,\ldots,\hat{\theta}^*_B$", "#a9e0a9"),
    (8.0,  "Distribucion\nBootstrap", "",                                            "#f7e1a0"),
    (11.3, "IC",                     "Percentiles\no BCa",                            "#f6c98a"),
]
for x, titulo, detalle, color in etapas:
    ax.add_patch(FancyBboxPatch((x - 1.15, 0.7), 2.3, 1.6,
                 boxstyle="round,pad=0.05", fc=color, ec="black", lw=1.5))
    ax.text(x, 1.85, titulo, ha="center", va="center", fontweight="bold", fontsize=10)
    if detalle:
        ax.text(x, 1.12, detalle, ha="center", va="center", fontsize=9)

for x0, x1, etq in [(2.55, 3.55, "remuestrear"), (5.85, 6.85, "agregar"), (9.15, 10.15, "calcular")]:
    ax.annotate("", xy=(x1, 1.5), xytext=(x0, 1.5),
                arrowprops=dict(arrowstyle="-|>", lw=2, color="#28527a"))
    ax.text((x0 + x1) / 2, 1.72, etq, ha="center", fontsize=8, style="italic")

ax.set_title("Del dato al intervalo de confianza", fontweight="bold")
plt.show()

> ### 📝 Lo que aprenderás
> Diferentes métodos bootstrap para IC (Normal, Percentil, BCa); su implementación desde cero; el uso de `scipy.stats.bootstrap`; la comparación con IC paramétricos tradicionales; cuándo usar cada método; sus ventajas y limitaciones; y la interpretación correcta de un IC bootstrap.

# 2. Repaso: Concepto de Intervalo de Confianza

> ### 📘 Definición — Intervalo de Confianza (IC)
> Un IC al nivel $1-\alpha$ es un rango $[L, U]$ tal que, si repetimos el experimento muchas veces, aproximadamente $(1-\alpha)\times 100\%$ de los intervalos calculados contendrán el parámetro verdadero $\theta$:
> $$P(L \leq \theta \leq U) = 1 - \alpha$$
> **Interpretación correcta:** es una frecuencia relativa en muestras repetidas, **no** la probabilidad de que $\theta$ esté en *este* intervalo particular ($\theta$ es fijo, el intervalo es aleatorio).

Esta idea es famosa por lo fácil que es malinterpretarla. En vez de un dibujo, **lo demostramos**: generamos 100 muestras de una población con media conocida $\mu=10$, calculamos el IC del 95% de cada una, y vemos cuántos atrapan a $\mu$.

In [ ]:
# Demostracion REAL del significado de un IC al 95%
mu_true, sigma, n = 10.0, 2.0, 30
n_sims = 100
rng = np.random.default_rng(4)
t_crit = stats.t.ppf(0.975, df=n - 1)

contiene = 0
plt.figure(figsize=(12, 5))
for i in range(n_sims):
    muestra = rng.normal(mu_true, sigma, n)
    m = muestra.mean()
    h = t_crit * muestra.std(ddof=1) / np.sqrt(n)        # semi-amplitud del IC
    dentro = (m - h) <= mu_true <= (m + h)
    contiene += int(dentro)
    color = "#2e7d32" if dentro else "#c62828"
    plt.errorbar(i, m, yerr=h, fmt="o", ms=3, color=color,
                 ecolor=color, elinewidth=1, capsize=2)

plt.axhline(mu_true, color="black", lw=1.6, ls="--", label=f"mu verdadero = {mu_true}")
plt.xlabel("Muestra simulada")
plt.ylabel("Media e IC 95%")
plt.title(f"{contiene} de {n_sims} intervalos contienen mu "
          f"({contiene/n_sims:.0%}, esperado ~95%)")
plt.legend(loc="upper right")
plt.show()

print(f"Cobertura empirica: {contiene}/{n_sims} = {contiene/n_sims:.1%}")
print("mu es FIJO; lo aleatorio es el intervalo. 'IC 95%' significa que en muestras")
print("repetidas ~95% de los intervalos atrapan a mu (NO es P(mu en ESTE intervalo)).")

# 3. Método 1: Bootstrap Normal

> ### 📘 Definición — Bootstrap Normal
> (1) Calcular $\hat{\theta}$ en los datos. (2) Generar $B$ remuestras y calcular $\hat{\theta}^*_1,\ldots,\hat{\theta}^*_B$. (3) Estimar el error estándar bootstrap:
> $$SE_{boot} = \sqrt{\tfrac{1}{B-1}\textstyle\sum_{b=1}^{B}(\hat{\theta}_b^* - \bar{\theta}^*)^2}$$
> (4) Construir el IC con cuantiles normales:
> $$IC_{1-\alpha} = \bigl[\hat{\theta} - z_{\alpha/2}\cdot SE_{boot},\; \hat{\theta} + z_{\alpha/2}\cdot SE_{boot}\bigr]$$
> **Ventajas:** simple, análogo a los IC paramétricos. **Desventajas:** asume simetría y un estimador aproximadamente insesgado; puede fallar con distribuciones asimétricas.

In [ ]:
def bootstrap_normal_ic(data, statistic=np.mean, n_bootstrap=10_000,
                        confidence=0.95, random_state=None):
    # IC bootstrap normal: theta_hat +/- z * SE_boot.
    # Devuelve: theta_hat, ic, se_boot, boot_dist
    rng  = np.random.default_rng(random_state)
    data = np.asarray(data)
    n    = len(data)

    theta_hat  = statistic(data)
    boot_stats = np.array([
        statistic(rng.choice(data, size=n, replace=True))
        for _ in range(n_bootstrap)
    ])

    se_boot = np.std(boot_stats, ddof=1)
    z_crit  = stats.norm.ppf(1 - (1 - confidence) / 2)
    ic      = (theta_hat - z_crit * se_boot, theta_hat + z_crit * se_boot)
    return theta_hat, ic, se_boot, boot_stats

# Datos asimetricos (exponencial) -> reutilizados en las secciones siguientes
rng  = np.random.default_rng(42)
data = rng.exponential(scale=5, size=50)

for stat, nombre in [(np.mean, "Media"), (np.median, "Mediana")]:
    th, ic, se, _ = bootstrap_normal_ic(data, stat, random_state=42)
    print(f"{nombre:8s}: {th:.4f}   SE={se:.4f}   IC 95%: [{ic[0]:.4f}, {ic[1]:.4f}]")

La esencia del bootstrap es **la distribución de los $\hat{\theta}^*$**. Visualicémosla para la media, con sus límites del IC normal marcados:

In [ ]:
th, ic, se, boot = bootstrap_normal_ic(data, np.mean, random_state=42)

plt.figure(figsize=(10, 5))
plt.hist(boot, bins=50, color="#9ecae1", edgecolor="white", density=True)
plt.axvline(th, color="black", lw=2, label=f"theta_hat = {th:.3f}")
plt.axvline(ic[0], color="#c62828", lw=2, ls="--",
            label=f"IC 95% normal: [{ic[0]:.3f}, {ic[1]:.3f}]")
plt.axvline(ic[1], color="#c62828", lw=2, ls="--")
plt.axvspan(ic[0], ic[1], color="#c62828", alpha=0.07)
plt.xlabel("Media de cada remuestra bootstrap")
plt.ylabel("Densidad")
plt.title("Distribucion bootstrap de la media (B = 10.000)")
plt.legend()
plt.show()

# 4. Método 2: Bootstrap Percentil

> ### 📘 Definición — Bootstrap Percentil
> (1) Generar $\hat{\theta}^*_1,\ldots,\hat{\theta}^*_B$. (2) Ordenarlos. (3) Usar los percentiles directamente como límites:
> $$IC_{1-\alpha} = \bigl[\hat{\theta}^*_{(\alpha/2)},\; \hat{\theta}^*_{(1-\alpha/2)}\bigr]$$
> Para $B=10{,}000$ y $\alpha=0.05$: el límite inferior está en la posición 250 y el superior en la 9{,}750.
>
> **Ventajas:** muy intuitivo, no asume simetría, respeta límites naturales (p.ej. varianza $\geq 0$). **Desventajas:** cobertura sub-óptima si hay sesgo; no corrige la asimetría del estimador.

In [ ]:
def bootstrap_percentil_ic(data, statistic=np.mean, n_bootstrap=10_000,
                           confidence=0.95, random_state=None):
    # IC bootstrap percentil: percentiles de la distribucion bootstrap.
    # Devuelve: theta_hat, ic, boot_dist
    rng  = np.random.default_rng(random_state)
    data = np.asarray(data)
    n    = len(data)

    theta_hat  = statistic(data)
    boot_stats = np.array([
        statistic(rng.choice(data, size=n, replace=True))
        for _ in range(n_bootstrap)
    ])

    alpha = 1 - confidence
    ic    = (np.percentile(boot_stats, 100 * alpha / 2),
             np.percentile(boot_stats, 100 * (1 - alpha / 2)))
    return theta_hat, ic, boot_stats

# Comparacion Normal vs Percentil para la mediana
th_n, ic_n, se_n, _ = bootstrap_normal_ic(data, np.median, random_state=42)
th_p, ic_p, _       = bootstrap_percentil_ic(data, np.median, random_state=42)

print("Mediana:")
print(f"  Normal:    IC [{ic_n[0]:.4f}, {ic_n[1]:.4f}]  amplitud={ic_n[1]-ic_n[0]:.4f}")
print(f"  Percentil: IC [{ic_p[0]:.4f}, {ic_p[1]:.4f}]  amplitud={ic_p[1]-ic_p[0]:.4f}")
print(f"  Diferencia limites: inf={abs(ic_n[0]-ic_p[0]):.4f}, sup={abs(ic_n[1]-ic_p[1]):.4f}")

# 5. Método 3: Bootstrap BCa

## 5.1 Idea y Fórmulas

> ### 📘 Definición — Bootstrap BCa (Bias-Corrected and accelerated)
> Ajusta los percentiles por dos parámetros.
>
> **1. Corrección de sesgo ($z_0$):** mide qué proporción de réplicas es menor que $\hat{\theta}$:
> $$z_0 = \Phi^{-1}\!\left(\frac{\#\{\hat{\theta}_b^* < \hat{\theta}\}}{B}\right)$$
>
> **2. Aceleración ($a$):** captura heterocedasticidad vía jackknife (donde $\hat{\theta}_{(i)}$ es el estadístico sin la observación $i$):
> $$a = \frac{\sum_{i=1}^{n}(\hat{\theta}_{(\cdot)} - \hat{\theta}_{(i)})^3}{6\left[\sum_{i=1}^{n}(\hat{\theta}_{(\cdot)} - \hat{\theta}_{(i)})^2\right]^{3/2}}$$
>
> **Percentiles ajustados:**
> $$\alpha_1 = \Phi\!\left(z_0 + \frac{z_0 + z_{\alpha/2}}{1 - a(z_0 + z_{\alpha/2})}\right), \qquad \alpha_2 = \Phi\!\left(z_0 + \frac{z_0 + z_{1-\alpha/2}}{1 - a(z_0 + z_{1-\alpha/2})}\right)$$
>
> **Ventajas:** mejor cobertura, corrige sesgo y asimetría. **Desventajas:** más costoso (jackknife); sensible a valores extremos.

## 5.2 Implementación desde Cero

In [ ]:
def bootstrap_bca_ic(data, statistic=np.mean, n_bootstrap=10_000,
                     confidence=0.95, random_state=None):
    # IC bootstrap BCa: corrige sesgo (z0) y aceleracion (a, via jackknife).
    # Devuelve: theta_hat, ic, boot_dist, z0, a
    rng  = np.random.default_rng(random_state)
    data = np.asarray(data)
    n    = len(data)

    theta_hat  = statistic(data)
    boot_stats = np.array([
        statistic(rng.choice(data, size=n, replace=True))
        for _ in range(n_bootstrap)
    ])

    # Correccion de sesgo
    prop = np.clip(np.mean(boot_stats < theta_hat), 1e-10, 1 - 1e-10)
    z0   = stats.norm.ppf(prop)

    # Aceleracion (jackknife: estadistico quitando una observacion a la vez)
    jack = np.array([statistic(np.delete(data, i)) for i in range(n)])
    jm   = jack.mean()
    num  = np.sum((jm - jack) ** 3)
    den  = 6 * np.sum((jm - jack) ** 2) ** 1.5
    a    = num / den if den != 0 else 0.0

    # Percentiles ajustados
    alpha   = 1 - confidence
    za2, z1 = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    a1      = stats.norm.cdf(z0 + (z0 + za2) / (1 - a * (z0 + za2)))
    a2      = stats.norm.cdf(z0 + (z0 + z1)  / (1 - a * (z0 + z1)))
    a1, a2  = np.clip(a1, 0, 1), np.clip(a2, 0, 1)

    ic = (np.percentile(boot_stats, 100 * a1),
          np.percentile(boot_stats, 100 * a2))
    return theta_hat, ic, boot_stats, z0, a

# Comparar los tres metodos (exponencial, media)
rng      = np.random.default_rng(42)
data_exp = rng.exponential(scale=5, size=50)

th_n, ic_n, se_n, _  = bootstrap_normal_ic(   data_exp, np.mean, random_state=42)
th_p, ic_p, _        = bootstrap_percentil_ic(data_exp, np.mean, random_state=42)
th_b, ic_b, _, z0, a = bootstrap_bca_ic(      data_exp, np.mean, random_state=42)

print(f"Media observada: {th_n:.4f}\n")
print(f"{'Metodo':>12}  {'IC inf':>8}  {'IC sup':>8}  {'Amplitud':>10}  {'Asimetria':>10}")
print("-" * 58)
for nombre, ic in [("Normal", ic_n), ("Percentil", ic_p), ("BCa", ic_b)]:
    amp  = ic[1] - ic[0]
    asim = abs((ic[1] - th_n) - (th_n - ic[0]))
    print(f"{nombre:>12}  {ic[0]:>8.4f}  {ic[1]:>8.4f}  {amp:>10.4f}  {asim:>10.4f}")
print(f"\nBCa  ->  z0 (sesgo) = {z0:.4f}   a (aceleracion) = {a:.4f}")

Veámoslo gráficamente: los **tres ICs sobre la misma distribución bootstrap**. En datos asimétricos, BCa desplaza sus límites respecto al percentil para corregir sesgo y asimetría.

In [ ]:
th, ic_b, boot, z0, a = bootstrap_bca_ic(data_exp, np.mean, random_state=42)
_, ic_n, _, _ = bootstrap_normal_ic(data_exp, np.mean, random_state=42)
_, ic_p, _    = bootstrap_percentil_ic(data_exp, np.mean, random_state=42)

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(boot, bins=50, color="#dfe6ee", edgecolor="white", density=True)
ax.axvline(th, color="black", lw=2, label=f"theta_hat = {th:.3f}")

ymax = ax.get_ylim()[1]
for nombre, ic, color, frac in [("Normal", ic_n, "#1f77b4", 0.88),
                                ("Percentil", ic_p, "#2ca02c", 0.74),
                                ("BCa", ic_b, "#d62728", 0.60)]:
    y = ymax * frac
    ax.plot([ic[0], ic[1]], [y, y], color=color, lw=3, solid_capstyle="round")
    ax.plot([ic[0], ic[1]], [y, y], "|", color=color, ms=15, mew=2)
    ax.text(ic[1], y, f"  {nombre}", va="center", color=color, fontweight="bold", fontsize=9)

ax.set_xlabel("Media bootstrap")
ax.set_ylabel("Densidad")
ax.set_title("Tres ICs sobre la misma distribucion bootstrap (datos asimetricos)")
ax.legend(loc="upper left")
plt.show()

# 6. Uso de `scipy.stats.bootstrap`

SciPy ofrece una implementación profesional y optimizada de los tres métodos.

> ### 🔧 Método — Interfaz de `scipy.stats.bootstrap`
> Parámetros clave: `method` (`'percentile'`, `'basic'`, `'BCa'`), `n_resamples` (default 9,999), `confidence_level`, y `vectorized` (acelera si el estadístico acepta arrays).

In [ ]:
# Interfaz profesional de scipy
res = bootstrap(
    (data_exp,),              # tupla de arrays (las muestras)
    statistic=np.mean,
    n_resamples=10_000,
    confidence_level=0.95,
    method="BCa",             # 'percentile', 'basic', 'BCa'
    random_state=42,
)
ic = res.confidence_interval
print(f"IC 95% (BCa, scipy): [{ic.low:.4f}, {ic.high:.4f}]")
print(f"SE bootstrap (scipy): {res.standard_error:.4f}")

¿Coinciden nuestras implementaciones manuales con SciPy? Comparémoslas sobre varios estadísticos (la pequeña diferencia $\Delta$ se debe a que cada uno usa su propio flujo aleatorio):

In [ ]:
def comparar_manual_scipy(data, statistic, nombre):
    # Compara nuestras implementaciones manuales con scipy (percentil y BCa)
    _, ic_pm, _       = bootstrap_percentil_ic(data, statistic, random_state=42)
    _, ic_bm, _, _, _ = bootstrap_bca_ic(      data, statistic, random_state=42)

    r_perc = bootstrap((data,), statistic, n_resamples=10_000,
                       method="percentile", random_state=np.random.default_rng(42))
    r_bca  = bootstrap((data,), statistic, n_resamples=10_000,
                       method="BCa", random_state=np.random.default_rng(42))
    ic_sp = (r_perc.confidence_interval.low, r_perc.confidence_interval.high)
    ic_sb = (r_bca.confidence_interval.low,  r_bca.confidence_interval.high)

    print(f"\n--- {nombre} = {statistic(data):.4f} ---")
    print(f"  Percentil manual:  [{ic_pm[0]:.4f}, {ic_pm[1]:.4f}]")
    print(f"  Percentil SciPy:   [{ic_sp[0]:.4f}, {ic_sp[1]:.4f}]  "
          f"d={max(abs(ic_pm[0]-ic_sp[0]), abs(ic_pm[1]-ic_sp[1])):.5f}")
    print(f"  BCa manual:        [{ic_bm[0]:.4f}, {ic_bm[1]:.4f}]")
    print(f"  BCa SciPy:         [{ic_sb[0]:.4f}, {ic_sb[1]:.4f}]  "
          f"d={max(abs(ic_bm[0]-ic_sb[0]), abs(ic_bm[1]-ic_sb[1])):.5f}")

rng       = np.random.default_rng(0)
data_test = rng.exponential(scale=5, size=100)
cv        = lambda x: np.std(x) / np.mean(x)     # coeficiente de variacion

comparar_manual_scipy(data_test, np.mean,   "Media")
comparar_manual_scipy(data_test, np.median, "Mediana")
comparar_manual_scipy(data_test, np.std,    "Desv. Estandar")
comparar_manual_scipy(data_test, cv,        "Coef. de Variacion")

# 7. Comparación con IC Paramétricos

> ### 📘 Definición — IC Paramétrico para la Media
> Bajo normalidad o con $n$ grande (TCL):
> $$IC_{1-\alpha} = \left[\bar{x} - t_{\alpha/2,\,n-1}\cdot\tfrac{s}{\sqrt{n}},\; \bar{x} + t_{\alpha/2,\,n-1}\cdot\tfrac{s}{\sqrt{n}}\right]$$
> **Supuestos:** datos normales o $n$ suficiente para el TCL; observaciones independientes.

Comparemos sistemáticamente el IC paramétrico con los tres métodos bootstrap sobre tres escenarios: datos **normales**, **asimétricos** y **con outliers**.

In [ ]:
def comparacion_completa(data, label=""):
    # Compara el IC parametrico (t de Student) con los tres metodos bootstrap
    n, media = len(data), np.mean(data)
    se_p     = np.std(data, ddof=1) / np.sqrt(n)
    t_crit   = stats.t.ppf(0.975, df=n - 1)
    ic_par   = (media - t_crit * se_p, media + t_crit * se_p)

    _, ic_n,  _, _    = bootstrap_normal_ic(   data, random_state=42)
    _, ic_pc, _       = bootstrap_percentil_ic(data, random_state=42)
    _, ic_bc, _, z0, a = bootstrap_bca_ic(     data, random_state=42)

    print(f"\n{'='*66}")
    print(f"DATOS: {label}  (n={n}, media={media:.4f})")
    print(f"{'='*66}")
    print(f"{'Metodo':>16}  {'Inf':>8}  {'Sup':>8}  {'Amplitud':>10}  {'Asimetria':>10}")
    print("-" * 66)
    for nom, ic in [("Parametrico", ic_par), ("Boot Normal", ic_n),
                    ("Boot Percentil", ic_pc), ("Boot BCa", ic_bc)]:
        amp  = ic[1] - ic[0]
        asim = abs((ic[1] - media) - (media - ic[0]))
        print(f"{nom:>16}  {ic[0]:>8.4f}  {ic[1]:>8.4f}  {amp:>10.4f}  {asim:>10.4f}")
    print(f"BCa: z0={z0:.4f}  a={a:.4f}")

rng = np.random.default_rng(42)
comparacion_completa(rng.normal(10, 2, 50),  label="Normal")
comparacion_completa(rng.exponential(5, 50), label="Exponencial (asimetrico)")
comparacion_completa(np.concatenate([rng.normal(10, 2, 45), [25, 26, 27, 28, 30]]),
                     label="Con outliers")

Fíjate cómo, en los datos normales, los cuatro métodos coinciden casi perfectamente (ahí el paramétrico es el más eficiente), mientras que en los asimétricos y con outliers el BCa muestra una **asimetría** distinta de cero: corre sus límites para reflejar la forma real de la distribución.

## 7.1 ¿Cuál cubre mejor de verdad? Cobertura empírica

La prueba de fuego de un IC del 95% es su **cobertura**: en muestras repetidas, ¿qué fracción atrapa al parámetro verdadero? Lo medimos sobre datos exponenciales (media verdadera conocida) con un $n$ pequeño, donde las diferencias entre métodos se notan.

In [ ]:
def cobertura_metodos(scale=5.0, n=25, n_trials=500, B=1500, seed=123):
    # Mide la fraccion de ICs (de cada metodo) que atrapa la media verdadera.
    # Bootstrap vectorizado (solo medias) + jackknife cerrado -> rapido.
    rng = np.random.default_rng(seed)
    mu_true = scale                                  # media verdadera de Exp(scale)
    za2, z1 = stats.norm.ppf(0.025), stats.norm.ppf(0.975)
    cuenta = {"Normal": 0, "Percentil": 0, "BCa": 0}

    for _ in range(n_trials):
        data = rng.exponential(scale, n)
        th   = data.mean()
        idx  = rng.integers(0, n, size=(B, n))       # remuestreo con reemplazo
        boot = data[idx].mean(axis=1)

        # Normal
        se = boot.std(ddof=1)
        ok_n = (th - z1 * se) <= mu_true <= (th + z1 * se)
        # Percentil
        lo_p, hi_p = np.percentile(boot, [2.5, 97.5])
        ok_p = lo_p <= mu_true <= hi_p
        # BCa
        prop = np.clip(np.mean(boot < th), 1e-10, 1 - 1e-10)
        z0   = stats.norm.ppf(prop)
        jack = (data.sum() - data) / (n - 1)          # jackknife de la media (cerrado)
        jm   = jack.mean()
        den  = 6 * np.sum((jm - jack) ** 2) ** 1.5
        a    = np.sum((jm - jack) ** 3) / den if den != 0 else 0.0
        a1   = stats.norm.cdf(z0 + (z0 + za2) / (1 - a * (z0 + za2)))
        a2   = stats.norm.cdf(z0 + (z0 + z1)  / (1 - a * (z0 + z1)))
        lo_b, hi_b = np.percentile(boot, [100 * np.clip(a1, 0, 1), 100 * np.clip(a2, 0, 1)])
        ok_b = lo_b <= mu_true <= hi_b

        cuenta["Normal"]    += int(ok_n)
        cuenta["Percentil"] += int(ok_p)
        cuenta["BCa"]       += int(ok_b)
    return {k: v / n_trials for k, v in cuenta.items()}

cov = cobertura_metodos()
print("Cobertura empirica objetivo: 95%  (media de Exp, n=25, 500 experimentos)\n")
for met, c in cov.items():
    print(f"  {met:10s}: {c:.1%}")

plt.figure(figsize=(7, 4.5))
nombres = list(cov.keys()); valores = [cov[k] * 100 for k in nombres]
barras = plt.bar(nombres, valores, color=["#1f77b4", "#2ca02c", "#d62728"])
plt.axhline(95, color="black", ls="--", lw=1.5, label="Nominal 95%")
plt.ylim(80, 100)
for b, v in zip(barras, valores):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}%", ha="center", fontsize=10)
plt.ylabel("Cobertura empirica (%)")
plt.title("Cobertura real de cada metodo (datos asimetricos, n=25)")
plt.legend()
plt.show()

# 👉 EXPERIMENTA: sube n (p.ej. 100) y veras que los tres metodos se acercan al 95%;
#    con n pequenio, Normal y Percentil tienden a quedar por debajo y BCa corrige mejor.

# 8. Guía de Decisión: ¿Qué Método Usar?

| Situación | Método recomendado |
|---|---|
| Datos normales, $n$ grande | IC paramétrico (más eficiente) |
| Datos normales, $n$ pequeño | IC paramétrico con $t$ de Student |
| Distribución desconocida, $n$ moderado | Bootstrap Percentil |
| Datos asimétricos | Bootstrap BCa o Percentil |
| Estadístico complejo (mediana, percentil) | Bootstrap BCa |
| Máxima precisión | Bootstrap BCa con $B \geq 10{,}000$ |
| Datos con outliers | Bootstrap con mediana + BCa |
| Rapidez como prioridad | IC paramétrico (si aplicable) o Boot Normal |

> ### ⭐ Recomendaciones Prácticas
> 1. **Exploración inicial:** bootstrap percentil con $B=1{,}000$. 2. **Análisis final:** BCa con $B \geq 10{,}000$. 3. **Reportar múltiples métodos:** si hay desacuerdo significativo, reporta ambos e investiga. 4. **Verificar supuestos:** si los datos son claramente normales y $n>30$, el IC paramétrico es más eficiente. 5. **Estadísticos complejos:** bootstrap es a menudo la única opción práctica. 6. **Documentar siempre:** el método y el número de réplicas $B$.

# 9. Limitaciones y Advertencias

> ### ⚠️ Cuándo Bootstrap Puede Fallar
> 1. **Muestra muy pequeña** ($n < 20$): la muestra original no representa bien la población; el bootstrap no crea información que no existe.
> 2. **Dependencia fuerte:** el bootstrap estándar asume independencia; para series temporales usa *block bootstrap*.
> 3. **Estimadores no suaves:** máximo/mínimo muestrales pueden tener mal desempeño.
> 4. **Colas pesadas:** distribuciones sin momentos finitos hacen que converja lentamente o falle.
> 5. **Pocas réplicas:** con $B$ pequeño los percentiles son inestables. Mínimo recomendado $B=1{,}000$; ideal $B \geq 10{,}000$.

# 10. Resumen y Puntos Clave

1. El bootstrap permite construir **IC sin supuestos paramétricos** sobre la distribución.
2. **Tres métodos principales:** Normal (SE bootstrap + cuantiles normales), Percentil (percentiles directos) y BCa (corrige sesgo y ajusta por aceleración).
3. **BCa** suele tener mejor cobertura que el percentil simple, especialmente con datos asimétricos.
4. `scipy.stats.bootstrap` ofrece una implementación profesional y optimizada de los tres métodos.
5. Para la **media con datos normales**, el IC paramétrico es más eficiente que el bootstrap.
6. El bootstrap es especialmente valioso para **estadísticos complejos** sin teoría analítica simple.
7. Réplicas: mínimo $B=1{,}000$ para exploración; $B \geq 10{,}000$ para resultados finales.
8. **Comparar múltiples métodos** ayuda a detectar problemas.
9. El bootstrap tiene **limitaciones**: muestra pequeña, dependencia, colas pesadas.

> ### 💡 Cierre
> El bootstrap transforma el problema de construir intervalos de confianza de uno de teoría matemática compleja a uno de simulación computacional. Los métodos Percentil y BCa proporcionan alternativas robustas a los IC paramétricos tradicionales, especialmente cuando los supuestos de normalidad fallan o cuando se trabaja con estadísticos complejos. La disponibilidad de implementaciones eficientes como `scipy.stats.bootstrap` hace estas técnicas accesibles y prácticas para cualquier análisis moderno de datos. Sin embargo, el bootstrap no es magia: requiere datos representativos y suficientes, y es fundamental conocer tanto sus fortalezas como sus limitaciones.